# Sfida: Analisi di un testo sulla Data Science

In questo esempio, facciamo un esercizio semplice che copre tutti i passaggi di un processo tradizionale di data science. Non devi scrivere alcun codice, puoi semplicemente cliccare sulle celle sottostanti per eseguirle e osservare il risultato. Come sfida, sei incoraggiato a provare questo codice con dati diversi. 

## Obiettivo

In questa lezione, abbiamo discusso diversi concetti legati alla Data Science. Cerchiamo di scoprire altri concetti correlati facendo un po' di **text mining**. Inizieremo con un testo sulla Data Science, ne estrarremo le parole chiave, e poi cercheremo di visualizzare il risultato.

Come testo, userò la pagina su Data Science di Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Passo 1: Ottenere i Dati

Il primo passo in ogni processo di data science è ottenere i dati. Useremo la libreria `requests` per farlo:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Passo 2: Trasformare i Dati

Il passo successivo è convertire i dati nella forma adatta per l'elaborazione. Nel nostro caso, abbiamo scaricato il codice sorgente HTML dalla pagina, e dobbiamo convertirlo in testo semplice.

Ci sono molti modi per farlo. Useremo [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), una popolare libreria Python per il parsing dell'HTML. BeautifulSoup ci permette di mirare a specifici elementi HTML, così possiamo concentrarci sul contenuto principale dell'articolo di Wikipedia e ridurre alcuni menu di navigazione, barre laterali, piè di pagina e altri contenuti irrilevanti (anche se qualche testo standard potrebbe ancora rimanere).


Per prima cosa, dobbiamo installare la libreria BeautifulSoup per l'analisi HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Passaggio 3: Ottenere informazioni

Il passaggio più importante è trasformare i nostri dati in una forma da cui possiamo estrarre informazioni. Nel nostro caso, vogliamo estrarre parole chiave dal testo e vedere quali parole chiave sono più significative.

Useremo la libreria Python chiamata [RAKE](https://github.com/aneesha/RAKE) per l'estrazione delle parole chiave. Prima, installiamo questa libreria nel caso non sia presente: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

La funzionalità principale è disponibile dall'oggetto `Rake`, che possiamo personalizzare utilizzando alcuni parametri. Nel nostro caso, imposteremo la lunghezza minima di una parola chiave a 5 caratteri, la frequenza minima di una parola chiave nel documento a 3 e il numero massimo di parole in una parola chiave a 2. Sentiti libero di sperimentare con altri valori e osservare il risultato.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Abbiamo ottenuto una lista di termini insieme al grado di importanza associato. Come puoi vedere, le discipline più rilevanti, come il machine learning e i big data, sono presenti nella lista nelle posizioni più alte.

## Passo 4: Visualizzare il Risultato

Le persone riescono a interpretare meglio i dati in forma visiva. Perciò, spesso ha senso visualizzare i dati per trarre alcune intuizioni. Possiamo usare la libreria `matplotlib` in Python per tracciare una semplice distribuzione delle parole chiave con la loro rilevanza:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Esiste, tuttavia, un modo ancora migliore per visualizzare la frequenza delle parole: utilizzare una **Word Cloud**. Dovremo installare un'altra libreria per tracciare la word cloud dalla nostra lista di parole chiave.


In [ ]:
!{sys.executable} -m pip install wordcloud

L'oggetto `WordCloud` è responsabile di prendere in input o il testo originale, o una lista pre-calcolata di parole con le loro frequenze, e restituisce un'immagine, che può poi essere mostrata utilizzando `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Possiamo anche fornire il testo originale a `WordCloud` - vediamo se siamo in grado di ottenere un risultato simile:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Puoi vedere che la word cloud ora appare più impressionante, ma contiene anche molto rumore (ad esempio parole non correlate come `Retrieved on`). Inoltre, otteniamo meno parole chiave costituite da due parole, come *data scientist* o *computer science*. Questo perché l'algoritmo RAKE fa un lavoro molto migliore nella selezione di buone parole chiave dal testo. Questo esempio illustra l'importanza della pre-elaborazione e pulizia dei dati, perché un quadro chiaro alla fine ci permetterà di prendere decisioni migliori.

In questo esercizio abbiamo seguito un processo semplice per estrarre un po' di significato dal testo di Wikipedia, sotto forma di parole chiave e word cloud. Questo esempio è piuttosto semplice, ma dimostra bene tutti i passaggi tipici che un data scientist seguirà lavorando con i dati, a partire dall'acquisizione dei dati fino alla visualizzazione.

Nel nostro corso discuteremo tutti questi passaggi in dettaglio.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
